# Module 1 — Data Curation

This notebook builds a structured dataset of endophyte-derived bioactive compounds from literature, then enriches each entry with molecular data fetched from the **PubChem REST API**.

**Run cells one by one from top to bottom. Do not skip any cell.**

In [ ]:
# ── CELL 1: Install packages (Colab already has pandas/matplotlib) ──
!pip install requests --quiet
print("✓ Packages ready")

In [ ]:
# ── CELL 2: Create folders ──
import os
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("✓ Folders created: data/ and figures/")

In [ ]:
# ── CELL 3: Import libraries ──
import pandas as pd
import requests
import time
import matplotlib.pyplot as plt
print("✓ Libraries imported")

In [ ]:
# ── CELL 4: Define compound dataset from literature ──
# Source: Banerjee (2019), Endophytes: Treasure Trove of Bioactive Molecules
# NTCC Review Article, Amity University Kolkata

compounds_raw = [
    {
        "compound_name": "Taxol",
        "pubchem_name": "Paclitaxel",
        "source_endophyte": "Taxomyces andreanae",
        "host_plant": "Taxus brevifolia",
        "endophyte_type": "Fungi",
        "bioactivity": "Anticancer",
        "target_disease": "Breast, lung, ovarian cancer",
        "fda_approved": True
    },
    {
        "compound_name": "Camptothecin",
        "pubchem_name": "Camptothecin",
        "source_endophyte": "Fusarium solani",
        "host_plant": "Nothapodytes foetida",
        "endophyte_type": "Fungi",
        "bioactivity": "Anticancer",
        "target_disease": "Colorectal, lung cancer",
        "fda_approved": False
    },
    {
        "compound_name": "Podophyllotoxin",
        "pubchem_name": "Podophyllotoxin",
        "source_endophyte": "Trametes hirsuta",
        "host_plant": "Podophyllum hexandrum",
        "endophyte_type": "Fungi",
        "bioactivity": "Anticancer",
        "target_disease": "Testicular cancer (precursor to etoposide)",
        "fda_approved": False
    },
    {
        "compound_name": "Ergoflavin",
        "pubchem_name": "Ergoflavin",
        "source_endophyte": "Endophytic fungi (unidentified)",
        "host_plant": "Mimusops elengi",
        "endophyte_type": "Fungi",
        "bioactivity": "Anticancer",
        "target_disease": "General cytotoxicity",
        "fda_approved": False
    },
    {
        "compound_name": "Pestacin",
        "pubchem_name": "Pestacin",
        "source_endophyte": "Pestalotiopsis microspora",
        "host_plant": "Terminalia morobensis",
        "endophyte_type": "Fungi",
        "bioactivity": "Antioxidant",
        "target_disease": "Oxidative stress-related diseases",
        "fda_approved": False
    },
    {
        "compound_name": "Griseofulvin",
        "pubchem_name": "Griseofulvin",
        "source_endophyte": "Xylaria sp.",
        "host_plant": "Cinchona pubescens",
        "endophyte_type": "Fungi",
        "bioactivity": "Antimicrobial",
        "target_disease": "Fungal skin infections",
        "fda_approved": True
    },
    {
        "compound_name": "Coronamycin",
        "pubchem_name": "Coronamycin",
        "source_endophyte": "Streptomyces sp.",
        "host_plant": "Monstera sp.",
        "endophyte_type": "Bacteria",
        "bioactivity": "Antimicrobial",
        "target_disease": "Cryptococcus, malaria",
        "fda_approved": False
    },
    {
        "compound_name": "Hypericin",
        "pubchem_name": "Hypericin",
        "source_endophyte": "Endophytic fungi (Hypericum)",
        "host_plant": "Hypericum perforatum",
        "endophyte_type": "Fungi",
        "bioactivity": "Antimicrobial",
        "target_disease": "Bacterial infections, antiviral",
        "fda_approved": False
    },
    {
        "compound_name": "Huperzine A",
        "pubchem_name": "Huperzine A",
        "source_endophyte": "Acremonium sp.",
        "host_plant": "Huperzia serrata",
        "endophyte_type": "Fungi",
        "bioactivity": "Neuroprotective",
        "target_disease": "Alzheimer's disease",
        "fda_approved": False
    },
    {
        "compound_name": "Subglutinol A",
        "pubchem_name": "Subglutinol A",
        "source_endophyte": "Fusarium subglutinans",
        "host_plant": "Tripterygium wilfordii",
        "endophyte_type": "Fungi",
        "bioactivity": "Immunosuppressive",
        "target_disease": "Autoimmune disease, transplant rejection",
        "fda_approved": False
    },
    {
        "compound_name": "Brefeldin A",
        "pubchem_name": "Brefeldin A",
        "source_endophyte": "Cladosporium sp.",
        "host_plant": "Quercus variabilis",
        "endophyte_type": "Fungi",
        "bioactivity": "Antimicrobial",
        "target_disease": "Fungal infections",
        "fda_approved": False
    },
    {
        "compound_name": "Kakadumycin A",
        "pubchem_name": "Kakadumycin A",
        "source_endophyte": "Streptomyces sp.",
        "host_plant": "Grevillea pteridifolia",
        "endophyte_type": "Bacteria",
        "bioactivity": "Antimicrobial",
        "target_disease": "Gram-positive bacteria, malaria",
        "fda_approved": False
    },
    {
        "compound_name": "Torreyanic acid",
        "pubchem_name": "Torreyanic acid",
        "source_endophyte": "Pestalotiopsis microspora",
        "host_plant": "Torreya taxifolia",
        "endophyte_type": "Fungi",
        "bioactivity": "Anticancer",
        "target_disease": "Selective cytotoxicity",
        "fda_approved": False
    },
    {
        "compound_name": "Emodin",
        "pubchem_name": "Emodin",
        "source_endophyte": "Diaporthe helianthi",
        "host_plant": "Hypericum perforatum",
        "endophyte_type": "Fungi",
        "bioactivity": "Antimicrobial",
        "target_disease": "Bacterial and fungal infections",
        "fda_approved": False
    },
    {
        "compound_name": "Helvolic acid",
        "pubchem_name": "Helvolic acid",
        "source_endophyte": "Cytonaema sp.",
        "host_plant": "Various",
        "endophyte_type": "Fungi",
        "bioactivity": "Antimicrobial",
        "target_disease": "Bacterial infections",
        "fda_approved": False
    }
]

df = pd.DataFrame(compounds_raw)
print(f"✓ Total compounds loaded: {len(df)}")
df.head()

In [ ]:
# ── CELL 5: Fetch data from PubChem API ──
# This cell makes ~15 API requests. Takes about 1 minute.

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def fetch_pubchem_data(compound_name):
    """
    Fetch CID, molecular formula, molecular weight, and SMILES
    from the PubChem REST API for a given compound name.
    """
    result = {
        "cid": None,
        "molecular_formula": None,
        "molecular_weight": None,
        "canonical_smiles": None,
        "iupac_name": None
    }
    try:
        url_cid = f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(compound_name)}/cids/JSON"
        r = requests.get(url_cid, timeout=10)
        if r.status_code != 200:
            print(f"  [not found] {compound_name}")
            return result
        cid = r.json()["IdentifierList"]["CID"][0]
        result["cid"] = cid

        props = "MolecularFormula,MolecularWeight,CanonicalSMILES,IUPACName"
        url_props = f"{PUBCHEM_BASE}/compound/cid/{cid}/property/{props}/JSON"
        r2 = requests.get(url_props, timeout=10)
        if r2.status_code == 200:
            prop_data = r2.json()["PropertyTable"]["Properties"][0]
            result["molecular_formula"] = prop_data.get("MolecularFormula")
            result["molecular_weight"]  = prop_data.get("MolecularWeight")
            result["canonical_smiles"]  = prop_data.get("CanonicalSMILES")
            result["iupac_name"]        = prop_data.get("IUPACName")
    except Exception as e:
        print(f"  [error] {compound_name}: {e}")
    return result


print("Fetching from PubChem API...\n")
pubchem_results = []
for _, row in df.iterrows():
    name = row["pubchem_name"]
    print(f"  Fetching: {name}")
    data = fetch_pubchem_data(name)
    pubchem_results.append(data)
    time.sleep(0.5)

print("\n✓ Done fetching!")

In [ ]:
# ── CELL 6: Merge and save to CSV ──
df_pubchem = pd.DataFrame(pubchem_results)
df_final   = pd.concat([df, df_pubchem], axis=1)

df_final.to_csv("data/compounds.csv", index=False)

found = df_final["cid"].notna().sum()
print(f"✓ Saved to data/compounds.csv")
print(f"  Compounds with PubChem data: {found} / {len(df_final)}")
df_final[["compound_name", "cid", "molecular_formula", "molecular_weight", "bioactivity"]]

In [ ]:
# ── CELL 7: Plot summary chart ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bioactivity distribution
bioactivity_counts = df_final["bioactivity"].value_counts()
axes[0].bar(bioactivity_counts.index, bioactivity_counts.values,
            color="#1565C0", edgecolor="white")
axes[0].set_title("Compounds by Bioactivity Category", fontweight="bold")
axes[0].set_xlabel("Bioactivity")
axes[0].set_ylabel("Number of compounds")
axes[0].tick_params(axis='x', rotation=30)

# Endophyte type
type_counts = df_final["endophyte_type"].value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index,
            autopct="%1.0f%%", colors=["#4C9BE8", "#7DC47D"], startangle=90)
axes[1].set_title("Endophyte Type Distribution", fontweight="bold")

plt.suptitle("Endophyte Bioactive Compounds — Overview", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figures/compound_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved to figures/compound_overview.png")

In [ ]:
# ── CELL 8: Download output files to your computer ──
from google.colab import files
files.download("data/compounds.csv")
files.download("figures/compound_overview.png")
print("✓ Files downloaded! Upload them to your GitHub repo.")